In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
 
B = "workspace.bronze"
S = "workspace.silver"

SALT = "msf-demo-salt"
 
NRIC_REGEX = r"^[ST][0-9]{7}[A-Z]$"
RAW_CLIENT_COLS = ["client_id", "nric", "full_name", "address", "region",
                   "household_income", "household_size", "updated_date"]
RAW_ASSIST_COLS = ["case_id", "client_id", "scheme_id", "application_date",
                   "assistance_amount", "status"]

In [0]:
"""Remove any earlier rows from this source file, then append."""
def write_batch(df, table, source_file):
    if spark.catalog.tableExists(table):
        spark.sql(f"DELETE FROM {table} WHERE source_file = '{source_file}'")
        df.write.mode("append").saveAsTable(table)
    else:
        df.write.mode("overwrite").saveAsTable(table)

"""Take rows that failed data-quality checks, preserve what they originally looked like, record why they failed and mark them as OPEN for remediation."""
def to_quarantine(v, raw_cols):
    return (
        v.filter(F.col("reason").isNotNull())
         .select(*raw_cols, "reason", "source_file", "ingest_ts")
         .withColumn("quarantine_ts", F.current_timestamp())
         .withColumn("dq_status", F.lit("OPEN"))
         .withColumn("remediated_ts", F.lit(None).cast("timestamp"))
)

In [0]:
"""Cleans and deduplicates scheme data before writing to the silver layer"""
def process_schemes(source_file):
    raw = spark.table(f"{B}.schemes").filter(F.col("source_file") == source_file)
    df = (
        raw.select(
            F.upper(F.trim("scheme_id")).alias("scheme_id"),
            F.trim("scheme_name").alias("scheme_name"),
            F.trim("category").alias("category"),
            "source_file", "ingest_ts")
        .filter(F.col("scheme_id").isNotNull() & F.col("scheme_name").isNotNull())
        .dropDuplicates(["scheme_id"])
    )
    df.write.mode("overwrite").saveAsTable(f"{S}.schemes")
    print(f"schemes: {df.count()} rows")
 
process_schemes("schemes.csv")

schemes: 8 rows


In [0]:
"""Standardise and validate client data and assign a failure reason if row does not meet the check."""
def validate_clients(df):
    typed = (
        df
        .withColumn("nric_clean", F.upper(F.trim("nric")))# fixes lowercase / spaces
        .withColumn("name_clean", F.trim("full_name")) # fixes whitespace
        .withColumn("address_clean", F.trim("address")) # fixes whitespace
        .withColumn("region_clean", F.trim("region")) # fixes whitespace
        .withColumn("client_id_int", F.expr("try_cast(trim(client_id) AS INT)")) # converts client id into int
        .withColumn("income_dec", F.expr("try_cast(trim(household_income) AS DECIMAL(10,2))")) # converts income into decimal
        .withColumn("size_int", F.expr("try_cast(trim(household_size) AS INT)")) # converts household size into int
        .withColumn("updated_dt", F.expr("try_cast(trim(updated_date) AS DATE)")) # converts updated date into date
    )
    nric_ok = F.coalesce(F.col("nric_clean").rlike(NRIC_REGEX), F.lit(False)) # checks for valid nric format
    return typed.withColumn(
        "reason",
        F.when(F.col("client_id_int").isNull(), "INVALID_CLIENT_ID")
         .when(~nric_ok, "INVALID_NRIC_FORMAT")
         .when(F.col("income_dec").isNull(), "NULL_OR_INVALID_INCOME")
         .when(F.col("income_dec") < 0, "NEGATIVE_INCOME")
         .when(F.col("size_int").isNull() | (F.col("size_int") < 1), "INVALID_HOUSEHOLD_SIZE")
         .when(F.col("updated_dt").isNull(), "INVALID_UPDATED_DATE")
    )
 
 
def process_clients(source_file):
    raw = spark.table(f"{B}.clients").filter(F.col("source_file") == source_file)
 
    # Ensure only one row per client id
    w = Window.partitionBy("client_id").orderBy("ingest_ts")
    raw = raw.withColumn("rn", F.row_number().over(w)).filter("rn = 1").drop("rn")
 
    v = validate_clients(raw)
 
    # Hash NRIC and Mask name
    good = v.filter(F.col("reason").isNull()).select(
        F.col("client_id_int").alias("client_id"),
        F.sha2(F.concat(F.col("nric_clean"), F.lit(SALT)), 256).alias("nric_hash"),
        F.regexp_replace("name_clean", r"(?<=\w)\w", "*").alias("full_name_masked"),
        F.col("address_clean").alias("address"),
        F.col("region_clean").alias("region"),
        F.col("income_dec").alias("household_income"),
        F.col("size_int").alias("household_size"),
        F.col("updated_dt").alias("updated_date"),
        "source_file", "ingest_ts",
    )
    bad = to_quarantine(v, RAW_CLIENT_COLS)
 
    write_batch(good, f"{S}.clients", source_file)
    write_batch(bad, f"{S}.quarantine_clients", source_file)
    print(f"clients [{source_file}]: silver={good.count()}, quarantined={bad.count()}")
 
process_clients("clients_day1.csv")

clients [clients_day1.csv]: silver=982, quarantined=18


In [0]:
"""Standardise and validate assistance data and assign a failure reason if row does not meet the check"""
def validate_assistance(df):
    # Needs silver.clients and silver.schemes to exist 
    clients = (spark.table(f"{S}.clients").select(F.col("client_id").alias("c_id")).distinct().withColumn("client_ok", F.lit(True)))
    schemes = (spark.table(f"{S}.schemes").select(F.col("scheme_id").alias("s_id")).distinct().withColumn("scheme_ok", F.lit(True)))
 
    s = F.lower(F.trim(F.col("status")))
    typed = (
        df
        .withColumn("case_id_int", F.expr("try_cast(trim(case_id) AS INT)")) 
        # converts case id into int
        .withColumn("client_id_int", F.expr("try_cast(trim(client_id) AS INT)")) # converts client id into int
        .withColumn("scheme_id_clean", F.upper(F.trim("scheme_id"))) # fixes lowercase / spaces
        .withColumn("app_date", F.expr("try_cast(trim(application_date) AS DATE)")) # converts application date into date
        .withColumn("amount_dec", F.expr("try_cast(trim(assistance_amount) AS DECIMAL(10,2))")) # converts amount into decimal
        
        # Status mapping rules: trim/lowercase, map known variants, unknown -> null (fails)
        .withColumn("status_clean",
                    F.when(s.isin("approved", "apprvd"), "Approved")
                     .when(s == "pending", "Pending")
                     .when(s == "rejected", "Rejected")
                     .when(s == "disbursed", "Disbursed"))
    )
    typed = (typed
             .join(clients, typed.client_id_int == clients.c_id, "left")
             .join(schemes, typed.scheme_id_clean == schemes.s_id, "left"))
 
    return typed.withColumn(
        "reason",
        F.when(F.col("case_id_int").isNull(), "INVALID_CASE_ID")
         .when(F.col("client_ok").isNull(), "ORPHAN_CLIENT")
         .when(F.col("scheme_ok").isNull(), "UNKNOWN_SCHEME")
         .when(F.col("amount_dec").isNull(), "NULL_OR_INVALID_AMOUNT")
         .when(F.col("amount_dec") < 0, "NEGATIVE_AMOUNT")
         .when(F.col("status_clean").isNull(), "INVALID_STATUS")
         .when(F.col("app_date").isNull(), "INVALID_APPLICATION_DATE")
         .when(F.col("app_date") > F.current_date(), "FUTURE_APPLICATION_DATE")
    )
 
"""Return a DataFrame with only the columns needed"""
def to_silver_assistance(v):
    return v.select(
        F.col("case_id_int").alias("case_id"),
        F.col("client_id_int").alias("client_id"),
        F.col("scheme_id_clean").alias("scheme_id"),
        F.col("app_date").alias("application_date"),
        F.col("amount_dec").alias("assistance_amount"),
        F.col("status_clean").alias("status"),
        "source_file", "ingest_ts",
    )
 
def process_assistance(source_file):
    raw = spark.table(f"{B}.assistance").filter(F.col("source_file") == source_file)
 
    w = Window.partitionBy("case_id").orderBy("ingest_ts")
    raw = raw.withColumn("rn", F.row_number().over(w)).filter("rn = 1").drop("rn")
 
    v = validate_assistance(raw)
    good = to_silver_assistance(v.filter(F.col("reason").isNull()))
    bad = to_quarantine(v, RAW_ASSIST_COLS)
 
    write_batch(good, f"{S}.assistance", source_file)
    write_batch(bad, f"{S}.quarantine_assistance", source_file)
    print(f"assistance [{source_file}]: silver={good.count()}, quarantined={bad.count()}")
 
process_assistance("assistance.csv")

assistance [assistance.csv]: silver=2355, quarantined=145


In [0]:
# Quarantine reasons summary 
display(spark.table(f"{S}.quarantine_clients").groupBy("reason").count())
display(spark.table(f"{S}.quarantine_assistance").groupBy("reason").count())

# Masking check: no raw NRIC, names partially masked
display(spark.table(f"{S}.clients").limit(10))

"""Checks if all rows in bronze are in silver and quarantine"""
def reconcile(bronze_tbl, silver_tbl, quarantine_tbl, key, source_file):
    bronze_total = spark.table(bronze_tbl).filter(F.col("source_file") == source_file).count()
    distinct_keys = (spark.table(bronze_tbl).filter(F.col("source_file") == source_file)
                     .select(key).distinct().count())
    in_silver = spark.table(silver_tbl).filter(F.col("source_file") == source_file).count()
    in_quar = (spark.table(quarantine_tbl)
               .filter((F.col("source_file") == source_file) & (F.col("dq_status") == "OPEN")).count())
    print(f"{source_file}: bronze={bronze_total}, duplicates removed={bronze_total - distinct_keys}, "
          f"silver={in_silver}, quarantine(open)={in_quar}, "
          f"balanced={distinct_keys == in_silver + in_quar}")
 
 
reconcile(f"{B}.clients", f"{S}.clients", f"{S}.quarantine_clients", "client_id", "clients_day1.csv")
reconcile(f"{B}.assistance", f"{S}.assistance", f"{S}.quarantine_assistance", "case_id", "assistance.csv")

reason,count
INVALID_NRIC_FORMAT,6
NULL_OR_INVALID_INCOME,6
NEGATIVE_INCOME,6


reason,count
NEGATIVE_AMOUNT,10
ORPHAN_CLIENT,101
UNKNOWN_SCHEME,10
INVALID_STATUS,6
FUTURE_APPLICATION_DATE,10
NULL_OR_INVALID_AMOUNT,8


client_id,nric_hash,full_name_masked,address,region,household_income,household_size,updated_date,source_file,ingest_ts
1,0b989923ec05f7fca0fc453dfcc2fa4d8d78d8d2805eb47ebf5eb3b135e2998f,W****** J*******,"Flat 32, Sian Streets",North,4827.00,3,2026-10-01,clients_day1.csv,2026-11-14T08:54:08.327Z
100,b2a5af535da372ab6fc546d5885ef25576331e1410eca77594c46a6b8a3238d0,M**** P*****,"Flat 96P, Hugh Walk",Central,3232.00,1,2026-10-01,clients_day1.csv,2026-11-14T08:54:08.327Z
1000,0d46e8bcad2efbc8147e800093b2649bdfb7521fec6812f93355ce42f6ba71e8,D***** M*****-H***,"Studio 9, Garry Branch",West,3517.00,7,2026-10-01,clients_day1.csv,2026-11-14T08:54:08.327Z
101,8536f5675b19b1ba792b7aca919fcc61b29c01f2fe22c6db573715f1082851ad,V****** P*****,"Studio 78c, Damian Roads",West,887.00,6,2026-10-01,clients_day1.csv,2026-11-14T08:54:08.327Z
102,2352ade62a53356aaab8d1d9d83f754d07670d3442bcc19bcac23b3e1f4edfb5,G***** T*****,11 Thompson Hills,Central,5205.00,7,2026-10-01,clients_day1.csv,2026-11-14T08:54:08.327Z
103,4681aa77ada22dac84789e1af1e24810d12f2c24b7b1bba48fd4473cf0962c18,G**** D*****,55 Lane Pines,Central,796.00,2,2026-10-01,clients_day1.csv,2026-11-14T08:54:08.327Z
104,6b714d43ad8cb07c71b544773cc1934658837166c96c8a8b7b1d3aa37af0f7c0,M* C**** J*****,"Studio 37, Dale Mountain",Central,4008.00,5,2026-10-01,clients_day1.csv,2026-11-14T08:54:08.327Z
105,2b82fb5b23074f55bbe5fde31a48a19ccab0987c5213b817e9487e2dffdea9f9,S**** N*******,48 Liam Mills,Central,5617.00,3,2026-10-01,clients_day1.csv,2026-11-14T08:54:08.327Z
106,128d8edad008303b276a21c866827698d53f9cc325c2cc7b33897745c6a45b3a,M*** F****** H***,"Studio 4, Jacqueline Walk",East,5256.00,6,2026-10-01,clients_day1.csv,2026-11-14T08:54:08.327Z
107,b955b0061f287a008c10066e294f82a93cc20b89ff3fa435ae1c5aea91f3384d,A******** S****,"Studio 44, Katy Locks",East,4803.00,2,2026-10-01,clients_day1.csv,2026-11-14T08:54:08.327Z


clients_day1.csv: bronze=1005, duplicates removed=5, silver=982, quarantine(open)=18, balanced=True
assistance.csv: bronze=2505, duplicates removed=5, silver=2355, quarantine(open)=145, balanced=True


In [0]:
# process clients for day 2
process_clients("clients_day2.csv")

reconcile(f"{B}.clients", f"{S}.clients", f"{S}.quarantine_clients", "client_id", "clients_day2.csv")

clients [clients_day2.csv]: silver=1010, quarantined=10
clients_day2.csv: bronze=1020, duplicates removed=0, silver=1010, quarantine(open)=10, balanced=True


In [0]:
""" Re-validate quarantined rows for assistance. Handles Orphan clients only """
def remediate_assistance():
    # Get the quarantined orphan records
    q = (spark.table(f"{S}.quarantine_assistance")
         .filter("dq_status = 'OPEN' AND reason = 'ORPHAN_CLIENT'")
         .withColumnRenamed("reason", "orig_reason"))
 
    # Revalidate the quarantined rows
    v = validate_assistance(q)
 
    # Don't re-insert cases that are already in silver
    done = spark.table(f"{S}.assistance").select(F.col("case_id").alias("done_id"))
    v = v.join(done, v.case_id_int == done.done_id, "left_anti")
 
    # Stage the rows that passes the validation check now
    (v.filter(F.col("reason").isNull())
      .write.mode("overwrite").saveAsTable(f"{S}.stg_assistance_remediated"))
    stg = spark.table(f"{S}.stg_assistance_remediated")
    n = stg.count()
 
    # add audit
    (stg.select(
        F.lit("assistance").alias("entity"),
        F.col("case_id_int").cast("string").alias("record_key"),
        F.col("orig_reason").alias("original_reason"),
        F.lit("REVALIDATED_AFTER_CLIENT_ARRIVAL").alias("action"),
        F.current_timestamp().alias("remediated_ts"))
     .write.mode("append").saveAsTable(f"{S}.remediation_audit"))
 
    # insert released rows into silver
    to_silver_assistance(stg).write.mode("append").saveAsTable(f"{S}.assistance")
 
    # close the quarantine records (kept for history, not deleted)
    spark.sql(f"""
        MERGE INTO {S}.quarantine_assistance t
        USING {S}.stg_assistance_remediated s
        ON t.case_id = s.case_id AND t.dq_status = 'OPEN'
        WHEN MATCHED THEN UPDATE SET t.dq_status = 'REMEDIATED', t.remediated_ts = current_timestamp()
    """)
 
    spark.sql(f"DROP TABLE IF EXISTS {S}.stg_assistance_remediated")
    print(f"released {n} orphan rows into silver")

In [0]:
remediate_assistance()
display(spark.table(f"{S}.remediation_audit"))

released 89 orphan rows into silver


entity,record_key,original_reason,action,remediated_ts
assistance,1041,ORPHAN_CLIENT,REVALIDATED_AFTER_CLIENT_ARRIVAL,2026-11-14T08:58:36.624Z
assistance,1054,ORPHAN_CLIENT,REVALIDATED_AFTER_CLIENT_ARRIVAL,2026-11-14T08:58:36.624Z
assistance,106,ORPHAN_CLIENT,REVALIDATED_AFTER_CLIENT_ARRIVAL,2026-11-14T08:58:36.624Z
assistance,1077,ORPHAN_CLIENT,REVALIDATED_AFTER_CLIENT_ARRIVAL,2026-11-14T08:58:36.624Z
assistance,1130,ORPHAN_CLIENT,REVALIDATED_AFTER_CLIENT_ARRIVAL,2026-11-14T08:58:36.624Z
assistance,1153,ORPHAN_CLIENT,REVALIDATED_AFTER_CLIENT_ARRIVAL,2026-11-14T08:58:36.624Z
assistance,1168,ORPHAN_CLIENT,REVALIDATED_AFTER_CLIENT_ARRIVAL,2026-11-14T08:58:36.624Z
assistance,1202,ORPHAN_CLIENT,REVALIDATED_AFTER_CLIENT_ARRIVAL,2026-11-14T08:58:36.624Z
assistance,1228,ORPHAN_CLIENT,REVALIDATED_AFTER_CLIENT_ARRIVAL,2026-11-14T08:58:36.624Z
assistance,1246,ORPHAN_CLIENT,REVALIDATED_AFTER_CLIENT_ARRIVAL,2026-11-14T08:58:36.624Z


In [0]:
"""Close quarantined client records when a LATER batch delivered a valid version of that client. """
def remediate_clients():
    q = (spark.table(f"{S}.quarantine_clients")
         .filter("dq_status = 'OPEN'")
         .withColumn("qid", F.expr("try_cast(trim(client_id) AS INT)"))
         .withColumn("q_date", F.expr("try_cast(trim(updated_date) AS DATE)")))
 
    good = (spark.table(f"{S}.clients")
            .select(F.col("client_id").alias("good_id"),
                    F.col("source_file").alias("good_file"),
                    F.col("updated_date").alias("good_date")))
 
    # A valid row for the same client, dated AFTER the rejected one, means it was corrected.
    resolved = (q.join(good, (F.col("qid") == good.good_id) & (good.good_date > F.col("q_date")), "inner")
                 .groupBy("client_id", "source_file", "reason")
                 .agg(F.expr("max_by(good_file, good_date)").alias("resolved_by_file"),
                      F.max("good_date").alias("resolved_by_date")))
 
    resolved.write.mode("overwrite").saveAsTable(f"{S}.stg_clients_remediated")
    stg = spark.table(f"{S}.stg_clients_remediated")
    n = stg.count()
 
    # add audit
    (stg.select(
        F.lit("clients").alias("entity"),
        F.col("client_id").alias("record_key"),
        F.col("reason").alias("original_reason"),
        F.lit("RESOLVED_BY_CORRECTED_RESEND").alias("action"),
        F.current_timestamp().alias("remediated_ts"))
     .write.mode("append").saveAsTable(f"{S}.remediation_audit"))
 
    # Close the quarantine records
    spark.sql(f"""
        MERGE INTO {S}.quarantine_clients t
        USING {S}.stg_clients_remediated s
        ON t.client_id = s.client_id AND t.source_file = s.source_file AND t.dq_status = 'OPEN'
        WHEN MATCHED THEN UPDATE SET t.dq_status = 'REMEDIATED', t.remediated_ts = current_timestamp()
    """)
 
    spark.sql(f"DROP TABLE IF EXISTS {S}.stg_clients_remediated")
    print(f"closed {n} client quarantine records (corrected version arrived in a later batch)")

In [0]:
remediate_clients()
display(spark.table(f"{S}.remediation_audit"))

closed 18 client quarantine records (corrected version arrived in a later batch)


entity,record_key,original_reason,action,remediated_ts
assistance,1041,ORPHAN_CLIENT,REVALIDATED_AFTER_CLIENT_ARRIVAL,2026-11-14T08:58:36.624Z
assistance,1054,ORPHAN_CLIENT,REVALIDATED_AFTER_CLIENT_ARRIVAL,2026-11-14T08:58:36.624Z
assistance,106,ORPHAN_CLIENT,REVALIDATED_AFTER_CLIENT_ARRIVAL,2026-11-14T08:58:36.624Z
assistance,1077,ORPHAN_CLIENT,REVALIDATED_AFTER_CLIENT_ARRIVAL,2026-11-14T08:58:36.624Z
assistance,1130,ORPHAN_CLIENT,REVALIDATED_AFTER_CLIENT_ARRIVAL,2026-11-14T08:58:36.624Z
assistance,1153,ORPHAN_CLIENT,REVALIDATED_AFTER_CLIENT_ARRIVAL,2026-11-14T08:58:36.624Z
assistance,1168,ORPHAN_CLIENT,REVALIDATED_AFTER_CLIENT_ARRIVAL,2026-11-14T08:58:36.624Z
assistance,1202,ORPHAN_CLIENT,REVALIDATED_AFTER_CLIENT_ARRIVAL,2026-11-14T08:58:36.624Z
assistance,1228,ORPHAN_CLIENT,REVALIDATED_AFTER_CLIENT_ARRIVAL,2026-11-14T08:58:36.624Z
assistance,1246,ORPHAN_CLIENT,REVALIDATED_AFTER_CLIENT_ARRIVAL,2026-11-14T08:58:36.624Z
